# Imports

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import OxfordIIITPet
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import classification_report
import random

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device: ' + str(device))

Using device: cpu


# Data

In [ ]:
# Since params inside ResNet18 is based of these values, we should use them and not the ones for oxford-IIIT
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

# Again 224 is needed since its what ResNet18 wants as input
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

# Data augmentation transform
train_transform_aug = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)), #crops + small size scaling (80% - 100% of original image)
    transforms.RandomHorizontalFlip(p=0.5),  #flip
    transforms.RandomRotation(15),               #small rotations (between -15 and 15 degrees)
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

# For binary (0,1)
train_set_binary = OxfordIIITPet(root='data', split='trainval', target_types='binary-category', transform=train_transform, download=True)
test_set_binary  = OxfordIIITPet(root='data', split='test', target_types='binary-category', transform=test_transform,  download=True)

train_loader_binary = DataLoader(train_set_binary, batch_size=50, shuffle=True,  num_workers=4) # Maybe batch size or/and num_workers should change value
test_loader_binary  = DataLoader(test_set_binary, batch_size=50, shuffle=False, num_workers=4)

# For full 37 (0-36)
train_set_full = OxfordIIITPet(root='data', split='trainval', target_types='category', transform=train_transform, download=True)
test_set_full  = OxfordIIITPet(root='data', split='test', target_types='category', transform=test_transform,  download=True)
train_set_full_aug = OxfordIIITPet(root='data', split='trainval', target_types='category', transform=train_transform_aug, download=True) #data aug train set


train_loader_full = DataLoader(train_set_full, batch_size=50, shuffle=True,  num_workers=4)
test_loader_full  = DataLoader(test_set_full, batch_size=50, shuffle=False, num_workers=4)
train_loader_full_aug = DataLoader(train_set_full_aug, batch_size=50, shuffle=True,  num_workers=4)


print(f'Train size for binary: {len(train_set_binary)}, Test size: {len(test_set_binary)}')
print(f'Train size for full: {len(train_set_full)}, Test size: {len(test_set_full)}')
print(f'Train size for full aug: {len(train_set_full_aug)}, Test size: {len(test_set_full)}')

Train size for binary: 3680, Test size: 3669
Train size for full: 3680, Test size: 3669
Train size for full aug: 3680, Test size: 3669


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


# Training for Binary Classification

In [ ]:
model = torchvision.models.resnet18(weights='IMAGENET1K_V1').to(device) # Only choice (of weights param)? # Added .to(device) to alow for T4 GPU, CPU still works /Björn

# Matching the tutorial
for param in model.parameters():
    param.requires_grad = False

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2).to(device)
model.to(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 148MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [ ]:
# Training for binary classification
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3) # Maybe lr should change for greater than 99%

num_epochs = 2

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0

    for images, labels in train_loader_binary:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()

    train_loss = running_loss / len(train_set_binary)
    train_acc  = correct / len(train_set_binary)


    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader_binary:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            # 0 = cat, 1 = dog for binary
            correct += (outputs.argmax(1) == labels).sum().item()

    test_acc = correct / len(test_set_binary)

    print(f'Epoch [{epoch+1:02d}/{num_epochs}]  Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  Test Acc: {test_acc:.4f}')

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


KeyboardInterrupt: 

In [ ]:
print('Final test accuracy for binary classification: ' + str(test_acc*100) + '%') # I get 97.7% after 2 epochs in vscode (~3min per epoch, takes a lot longer in colab)

# Training for Full Classification

In [ ]:
model = torchvision.models.resnet18(weights='IMAGENET1K_V1').to(device) # Only choice (of weights param)?

# Matching the tutorial
for param in model.parameters():
    param.requires_grad = False

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 37).to(device)
model.to(device)

In [ ]:
# Training for full classification
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3) # Maybe lr should change for greater than 99%

num_epochs = 2

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0

    for images, labels in train_loader_full:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()

    train_loss = running_loss / len(train_set_full)
    train_acc  = correct / len(train_set_full)


    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader_full:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            # 0-36 depending on breed
            correct += (outputs.argmax(1) == labels).sum().item()

    test_acc = correct / len(test_set_full)

    print(f'Epoch [{epoch+1:02d}/{num_epochs}]  Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  Test Acc: {test_acc:.4f}')

In [ ]:
print('Final test accuracy for full classification: ' + str(test_acc*100) + '%') # I get 84.4% after 2 epochs in vscode (~3min per epoch, takes a lot longer in colab)



```
# This is formatted as code
```

# Strategy 1

Unfreeze layers

In [ ]:
"""model = torchvision.models.resnet18(weights='IMAGENET1K_V1') # Only choice (of weights param)?

# Matching the tutorial
for param in model.parameters():
    param.requires_grad = False

trainable_blocks = []

for name, module in model.named_children():
    has_params = any(p.requires_grad is not None for p in module.parameters())

    if has_params:
        trainable_blocks.append((name, module))

l = 3 # number of layers to unfreeze

last_blocks = trainable_blocks[-(l+1):-1]



for name, block in last_blocks:
    for param in block.parameters():
        param.requires_grad = True

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 37).to(device)
model = model.to(device)

"""

In [ ]:
"""# Training for full classification
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4) # Maybe lr should change for greater than 99%

num_epochs = 2

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0

    for images, labels in train_loader_full:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()

    train_loss = running_loss / len(train_set_full)
    train_acc  = correct / len(train_set_full)


    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader_full:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            # 0-36 depending on breed
            correct += (outputs.argmax(1) == labels).sum().item()

    test_acc = correct / len(test_set_full)

    print(f'Epoch [{epoch+1:02d}/{num_epochs}]  Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  Test Acc: {test_acc:.4f}')"""

In [ ]:
print('Final test accuracy for full classification with one unefreezed layer: ' + str(test_acc*100) + '%') # I get 86.9, 88.8, and 87,8% for 1, 2, 3 layers repsectively taking ~7, 8, 10 minutes

# Strategy 2

In [ ]:
# help funtion
def unfreeze_last_blocks(trainable_blocks, l):

    if l == 0:
        print('Only fc is trainable')
        return

    last_blocks = trainable_blocks[-(l+1):-1]

    for name, block in last_blocks:
        for param in block.parameters():
            param.requires_grad = True

    print('Unfrozen blocks:', [name for name, _ in last_blocks])

# choose layers to unfreeze

l = 2 # choose amount of unfrozen layers
stages = list(range(1,l+1))

In [ ]:
model = torchvision.models.resnet18(weights='IMAGENET1K_V1').to(device) # Only choice (of weights param)?

# Matching the tutorial
for param in model.parameters():
    param.requires_grad = False

trainable_blocks = []

for name, module in model.named_children():
    has_params = any(p.requires_grad is not None for p in module.parameters())

    if has_params:
        trainable_blocks.append((name, module))

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 37).to(device)
model = model.to(device)


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3) # Maybe lr should change for greater than 99%

num_epochs = 2

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0

    for images, labels in train_loader_full:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()

    train_loss = running_loss / len(train_set_full)
    train_acc  = correct / len(train_set_full)


for stage in stages:
    criterion = nn.CrossEntropyLoss()
    unfreeze_last_blocks(trainable_blocks, stage)

    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0

        for images, labels in train_loader_full:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()

        train_loss = running_loss / len(train_set_full)
        train_acc  = correct / len(train_set_full)



model.eval()
correct = 0
with torch.no_grad():
    for images, labels in test_loader_full:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        # 0-36 depending on breed
        correct += (outputs.argmax(1) == labels).sum().item()

test_acc = correct / len(test_set_full)

KeyboardInterrupt: 

In [ ]:
print('Final test accuracy for full classification with two gradually unefreezed layer: ' + str(test_acc*100) + '%') # I get 90.0% doing gradual unfreezing with two layers (~11 min total, takes a lot longer in colab)


# Fine-tuning with limited data

## full classification with label fractions

In [ ]:
from torch.utils.data import Subset, DataLoader
from collections import defaultdict
import random

def stratified_subset(dataset, fraction, seed=42):
    random.seed(seed)
    class_to_indices = defaultdict(list)

    for idx in range(len(dataset)):
        label = dataset[idx][1]
        class_to_indices[label].append(idx)

    selected_indices = []

    for label, indices in class_to_indices.items():
        random.shuffle(indices)
        n_keep = max(1, int(len(indices) * fraction))
        selected_indices.extend(indices[:n_keep])

    random.shuffle(selected_indices)

    return Subset(dataset, selected_indices)

In [ ]:
train_set_100 = stratified_subset(train_set_full, 1.00)
train_set_10  = stratified_subset(train_set_full, 0.10)
train_set_1   = stratified_subset(train_set_full, 0.01)

batch_size = 50
train_loader_100 = DataLoader(train_set_100, batch_size=batch_size, shuffle=True, num_workers=2)
train_loader_10  = DataLoader(train_set_10,  batch_size=batch_size, shuffle=True, num_workers=2)
train_loader_1   = DataLoader(train_set_1,   batch_size=batch_size, shuffle=True, num_workers=2)

print("100% training size:", len(train_set_100))
print("10% training size:", len(train_set_10))
print("1% training size:", len(train_set_1))

KeyboardInterrupt: 

In [ ]:
#Testing full classification with the different data sizes
def modelGeneration():
    model = torchvision.models.resnet18(weights='IMAGENET1K_V1').to(device) # Only choice (of weights param)?

    # Matching the tutorial
    for param in model.parameters():
        param.requires_grad = False

    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, 37).to(device)
    model.to(device)

    return model

In [ ]:

def fullClassification(model, train_loader, lr): #lr should be 1e-3
    # Training for full classification

    #print(f'Results for fraction: {(len(train_loader.dataset)/3680):.2f}')

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.fc.parameters(), lr=lr) # Maybe lr should change for greater than 99%

    num_epochs = 2

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()


        train_size = len(train_loader.dataset)

        train_loss = running_loss / train_size
        train_acc  = correct / train_size



        model.eval()
        correct = 0
        with torch.no_grad():
            for images, labels in test_loader_full:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                # 0-36 depending on breed
                correct += (outputs.argmax(1) == labels).sum().item()

        test_acc = correct / len(test_set_full)

        print(f'Epoch [{epoch+1:02d}/{num_epochs}]  Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  Test Acc: {test_acc:.4f}')
    return test_acc

In [ ]:

print(f'Final test accuracy for full classification for different fractions of labelled data')
lr = 1e-3
model = modelGeneration()
test_acc = fullClassification(model, train_loader_1, lr)
print(f'Fraction: {(len(train_loader_1.dataset)/3680):.2f}: {test_acc*100:.3f}%')
model = modelGeneration()
#test_acc = fullClassification(model, train_loader_10, lr)
print(f'Fraction: {(len(train_loader_10.dataset)/3680):.2f}: {test_acc*100:.3f}%')
model = modelGeneration()
#test_acc = fullClassification(model, train_loader_100, lr)
print(f'Fraction: {(len(train_loader_100.dataset)/3680):.2f}: {test_acc*100:.3f}%')




Final test accuracy for full classification for different fractions of labelled data


KeyboardInterrupt: 

## Fine tune with fractions strategy 1

In [ ]:
def modelGenUnfreeze(numUnfreezeLayers):
    model = torchvision.models.resnet18(weights='IMAGENET1K_V1') # Only choice (of weights param)?

    # Matching the tutorial
    for param in model.parameters():
        param.requires_grad = False

    trainable_blocks = []

    for name, module in model.named_children():
        has_params = any(p.requires_grad is not None for p in module.parameters())

        if has_params:
            trainable_blocks.append((name, module))

    l = numUnfreezeLayers

    last_blocks = trainable_blocks[-(l+1):-1]

    for name, block in last_blocks:
        for param in block.parameters():
            param.requires_grad = True

    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, 37).to(device)
    model = model.to(device)
    return model

In [ ]:
lr = 1e-4
model = modelGenUnfreeze(3)
test_acc = fullClassification(model, train_loader_1, lr)
model = modelGenUnfreeze(3)
#test_acc = fullClassification(model, train_loader_10, lr)
#model = modelGenUnfreeze()
#test_acc = fullClassification(model, train_loader_100, lr)


KeyboardInterrupt: 

## Strategy 2 (gradual unfreezing of layers) with different fractions of labelled data

In [ ]:
def modelGenAndfullClassGradUnfreeze(train_loader, l):  #OBS, l is not lr
  model = torchvision.models.resnet18(weights='IMAGENET1K_V1').to(device) # Only choice (of weights param)?

  # Matching the tutorial
  for param in model.parameters():
      param.requires_grad = False

  trainable_blocks = []

  for name, module in model.named_children():
      has_params = any(p.requires_grad is not None for p in module.parameters())

      if has_params:
          trainable_blocks.append((name, module))

  num_features = model.fc.in_features
  model.fc = nn.Linear(num_features, 37).to(device)
  model = model.to(device)

  stages = list(range(1,l+1))

  criterion = nn.CrossEntropyLoss()
  optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3) # Maybe lr should change for greater than 99%

  num_epochs = 2

  for epoch in range(num_epochs):
      model.train()
      running_loss = 0.0
      correct = 0

      for images, labels in train_loader:
          images, labels = images.to(device), labels.to(device)

          optimizer.zero_grad()
          outputs = model(images)
          loss = criterion(outputs, labels)
          loss.backward()
          optimizer.step()

          running_loss += loss.item() * images.size(0)
          correct += (outputs.argmax(1) == labels).sum().item()

      train_size = len(train_loader.dataset)
      train_loss = running_loss / train_size
      train_acc  = correct / train_size


  for stage in stages:
      criterion = nn.CrossEntropyLoss()
      unfreeze_last_blocks(trainable_blocks, stage)

      optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4) #reasonable to use different learning rate here

      for epoch in range(num_epochs):
          model.train()
          running_loss = 0.0
          correct = 0

          for images, labels in train_loader:
              images, labels = images.to(device), labels.to(device)

              optimizer.zero_grad()
              outputs = model(images)
              loss = criterion(outputs, labels)
              loss.backward()
              optimizer.step()

              running_loss += loss.item() * images.size(0)
              correct += (outputs.argmax(1) == labels).sum().item()

          train_size = len(train_loader.dataset)
          train_loss = running_loss / train_size
          train_acc  = correct / train_size



  model.eval()
  correct = 0
  with torch.no_grad():
      for images, labels in test_loader_full:
          images, labels = images.to(device), labels.to(device)
          outputs = model(images)
          # 0-36 depending on breed
          correct += (outputs.argmax(1) == labels).sum().item()

  test_acc = correct / len(test_set_full)
  return test_acc

In [ ]:

print(f'Final test accuracy for full classification with strategy 2 for different fractions of labelled data')
# choose layers to unfreeze
l = 2 # choose amount of unfrozen layers
test_acc = modelGenAndfullClassGradUnfreeze(train_loader_1, l)
print(f'Fraction: {(len(train_loader_1.dataset)/3680):.2f}: {test_acc*100:.3f}%')

Final test accuracy for full classification with strategy 2 for different fractions of labelled data
Unfrozen blocks: ['layer4']


KeyboardInterrupt: 

In [ ]:
from torch.utils.data import Subset, DataLoader
from collections import defaultdict
import random

#returns 2 fraction subsets with the same data split, one with no data aug, and one with data aug
def stratified_subset_noaug_and_aug(dataset_noaug, dataset_aug, fraction, seed=42):
    random.seed(seed)
    class_to_indices = defaultdict(list)

    for idx in range(len(dataset_noaug)):
        label = dataset_noaug[idx][1]
        class_to_indices[label].append(idx)

    selected_indices = []

    for label, indices in class_to_indices.items():
        random.shuffle(indices)
        n_keep = max(1, int(len(indices) * fraction))
        selected_indices.extend(indices[:n_keep])

    random.shuffle(selected_indices)

    return Subset(dataset_noaug, selected_indices), Subset(dataset_aug, selected_indices)

In [ ]:
train_set_100_noaug, train_set_100_aug = stratified_subset_noaug_and_aug(train_set_full,train_set_full_aug, 1.00) #both the aug and noaug set have the same datasplit
train_set_10_noaug, train_set_10_aug  = stratified_subset_noaug_and_aug(train_set_full, train_set_full_aug, 0.10)
train_set_1_noaug, train_set_1_aug   = stratified_subset_noaug_and_aug(train_set_full, train_set_full_aug, 0.01)

batch_size = 50
train_loader_100_noaug = DataLoader(train_set_100_noaug, batch_size=batch_size, shuffle=True, num_workers=2)
train_loader_10_noaug  = DataLoader(train_set_10_noaug,  batch_size=batch_size, shuffle=True, num_workers=2)
train_loader_1_noaug   = DataLoader(train_set_1_noaug,   batch_size=batch_size, shuffle=True, num_workers=2)
train_loader_100_aug = DataLoader(train_set_100_aug, batch_size=batch_size, shuffle=True, num_workers=2)
train_loader_10_aug  = DataLoader(train_set_10_aug,  batch_size=batch_size, shuffle=True, num_workers=2)
train_loader_1_aug   = DataLoader(train_set_1_aug,   batch_size=batch_size, shuffle=True, num_workers=2)

print("noaug sets")
print("100% training size:", len(train_set_100_noaug))
print("10% training size:", len(train_set_10_noaug))
print("1% training size:", len(train_set_1_noaug))

print("aug sets")
print("100% training size:", len(train_set_100_aug))
print("10% training size:", len(train_set_10_aug))
print("1% training size:", len(train_set_1_aug))


noaug sets
100% training size: 3680
10% training size: 365
1% training size: 37
aug sets
100% training size: 3680
10% training size: 365
1% training size: 37


In [ ]:
print(f'Final test accuracy for full classification for different fractions of labelled data with and without data augmentation')

lr = 1e-3

model = modelGeneration()
test_acc = fullClassification(model, train_loader_1_noaug, lr)
print(f'Fraction without dataaug: {(len(train_loader_1_noaug.dataset)/3680):.2f}: {test_acc*100:.3f}%')

model = modelGeneration()
test_acc = fullClassification(model, train_loader_1_aug, lr)
print(f'Fraction with dataaug: {(len(train_loader_1_aug.dataset)/3680):.2f}: {test_acc*100:.3f}%')

model = modelGeneration()
#test_acc = fullClassification(model, train_loader_10_noaug, lr)
print(f'Fraction without dataaug: {(len(train_loader_10_noaug.dataset)/3680):.2f}: {test_acc*100:.3f}%')

model = modelGeneration()
#test_acc = fullClassification(model, train_loader_10_aug, lr)
print(f'Fraction with dataaug: {(len(train_loader_10_aug.dataset)/3680):.2f}: {test_acc*100:.3f}%')

model = modelGeneration()
#test_acc = fullClassification(model, train_loader_100_noaug, lr)
print(f'Fraction without dataaug: {(len(train_loader_100_noaug.dataset)/3680):.2f}: {test_acc*100:.3f}%')

model = modelGeneration()
#test_acc = fullClassification(model, train_loader_100_aug, lr)
print(f'Fraction with dataaug: {(len(train_loader_100_aug.dataset)/3680):.2f}: {test_acc*100:.3f}%')


Final test accuracy for full classification for different fractions of labelled data with and without data augmentation
Epoch [01/2]  Loss: 3.8827  Train Acc: 0.0000  Test Acc: 0.0262




# Fine-tuning with imbalanced classes

In [ ]:
cats = [] # contains the labels that has cats
for i, breed in enumerate(train_set_full.classes):
  if breed[0].isupper():
    cats.append(i) # This works since all cat breeds start with an uppercase letter in the Oxford-IIIT-pet dataset, whereas dogs start with lowercase

indices_for_kept_images = []
percentage_kept = 0.20

for index in range(len(train_set_full)):
  label = train_set_full[index][1]
  if label < 12:
    if random.random() < percentage_kept:
      indices_for_kept_images.append(index)
  else:
    indices_for_kept_images.append(index)

train_set_imbalanced = Subset(train_set_full, indices_for_kept_images)
train_loader_imbalanced = DataLoader(train_set_imbalanced, batch_size=50, shuffle=True,  num_workers=2) # Choosing num_workers=2 when running in Colab


#  Investigate semi-supervised learning to incorporate unlabelled data when labelled training data is limited

### Function to generate fractions of labelled and unlabelled data

In [ ]:
from torch.utils.data import Subset, DataLoader
from collections import defaultdict
import random

#Returns split of training data with fraction labelled data and (1 - fraction) unlabelled data
def stratified_labelled_unlabelled_split(dataset, fraction, seed=42):
    random.seed(seed)
    class_to_indices = defaultdict(list)

    for idx in range(len(dataset)):
        label = dataset[idx][1]
        class_to_indices[label].append(idx)

    labelled_data_indices = []
    unlabelled_data_indices = []

    for label, indices in class_to_indices.items():
        random.shuffle(indices)
        n_keep = max(1, int(len(indices) * fraction))
        labelled_data_indices.extend(indices[:n_keep]) #adds first n_keep indices
        unlabelled_data_indices.extend(indices[n_keep:]) #adds all indices after first n_keep indices => different indices (images) than the labelled data


    random.shuffle(labelled_data_indices)
    random.shuffle(unlabelled_data_indices)

    return Subset(dataset, labelled_data_indices), Subset(dataset, unlabelled_data_indices)

In [ ]:
#Creates the labelled and unlabelled fraction splits

label_set_100,unlabel_set_0  = stratified_labelled_unlabelled_split(train_set_full, 1.00)
label_set_50,unlabel_set_50  = stratified_labelled_unlabelled_split(train_set_full, 0.50)
label_set_10,unlabel_set_90  = stratified_labelled_unlabelled_split(train_set_full, 0.10)
label_set_1,unlabel_set_99   = stratified_labelled_unlabelled_split(train_set_full, 0.01)

batch_size = 50

label_loader_100 = DataLoader(label_set_100, batch_size=batch_size, shuffle=True, num_workers=2)
label_loader_50  = DataLoader(label_set_50,  batch_size=batch_size, shuffle=True, num_workers=2)
label_loader_10  = DataLoader(label_set_10,  batch_size=batch_size, shuffle=True, num_workers=2)
label_loader_1   = DataLoader(label_set_1,   batch_size=batch_size, shuffle=True, num_workers=2)

#unlabel_loader_0 = DataLoader(unlabel_set_0, batch_size=batch_size, shuffle=True, num_workers=2)
unlabel_loader_50  = DataLoader(unlabel_set_50,  batch_size=batch_size, shuffle=True, num_workers=2)
unlabel_loader_90  = DataLoader(unlabel_set_90,  batch_size=batch_size, shuffle=True, num_workers=2)
unlabel_loader_99   = DataLoader(unlabel_set_99,   batch_size=batch_size, shuffle=True, num_workers=2)

print(" labelled 100% training size:", len(label_set_100))
print("labelled 50% training size:", len(label_set_50))
print("labelled 10% training size:", len(label_set_10))
print("labelled 1% training size:", len(label_set_1))

print(" unlabelled 0% training size:", len(unlabel_set_0))
print("unlabelled 50% training size:", len(unlabel_set_50))
print("unlabelled 90% training size:", len(unlabel_set_90))
print("unlabelled 99% training size:", len(unlabel_set_99))

### Baseline for the semi-supervised learning part

In [ ]:
print(f'Final test accuracy for full classification when using different fractions of labelled data and no unlabelled data')
lr = 1e-3
model = modelGeneration()
test_acc = fullClassification(model, label_loader_1, lr)
print(f'Fraction: {(len(label_loader_1.dataset)/3680):.2f}: {test_acc*100:.3f}%')
model = modelGeneration()
test_acc = fullClassification(model, label_loader_10, lr)
print(f'Fraction: {(len(label_loader_10.dataset)/3680):.2f}: {test_acc*100:.3f}%')
model = modelGeneration()
test_acc = fullClassification(model, label_loader_50, lr)
print(f'Fraction: {(len(label_loader_50.dataset)/3680):.2f}: {test_acc*100:.3f}%')
model = modelGeneration()
test_acc = fullClassification(model, label_loader_100, lr)
print(f'Fraction: {(len(label_loader_100.dataset)/3680):.2f}: {test_acc*100:.3f}%')
